# Window Functions
# 

In [0]:
# row number - adds the seriol number to the dataframe

# Dense rank - it will also adds serial number but with strict sequence like giving same number to same value/same duplicate value

# RANK - it adds a serial number where duplicate values get the same number, but it skips the next numbers in the sequence to account for the ties.

# --- LEAD and LAG are used to look forward or backward in your dataset without changing the row order. ---
#lead - 
#lag - 




In [0]:
from pyspark.sql import functions as F

In [0]:
order_df = spark.table("samples.tpch.orders")

order_df.display()

In [0]:
order_df_window = order_df.withColumn("row_number",F.col("o_orderkey")).sort(F.col("row_number"))

order_df_window.display()

In [0]:
order_df_window = order_df.withColumn("dense_rank",F.col("o_custkey")).sort(F.col("dense_rank"))
order_df_window.display()

In [0]:
cust_df =  spark.table("samples.tpch.customer")
cust_df.show()

In [0]:
#Creating TempView to write sql comands on the dataframe

order_df.createTempView("orders")

In [0]:
# I need find Last order of the customer


In [0]:
%sql
select * from orders;

In [0]:
%sql
-- # Solving Problem using sql
select o_custkey, o_orderkey, max(o_orderDate) as max_order_date
from orders 
group by o_custkey , o_orderkey 
order by o_custkey desc , o_orderkey desc;

In [0]:
# USING WINDOW FUNCTIOSN

from pyspark.sql.window import Window as W
from pyspark.sql import functions as F




In [0]:
windoe_spec = W.partitionBy("o_custkey").orderBy(F.desc("o_orderdate"))
ran_ordet_df = order_df.withColumn("rank",F.dense_rank().over(windoe_spec))

display(ran_ordet_df)


In [0]:
ran_ordet_df = (
    ran_ordet_df
    .filter(F.col("rank") == 1)
    .drop("rank")
)
ran_ordet_df.display()

In [0]:
%sql
select * from orders  
qualify dense_rank() over (partition by o_custkey order by o_orderdate desc) =1;

In [0]:
query = """ 
select * from orders  
qualify dense_rank() over (partition by o_custkey order by o_orderdate desc) =1;
"""

ran_ordet_df = spark.sql(query)

display(ran_ordet_df)


